In [1]:
import pandas as pd
import umap
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load the datasets
print("Loading datasets...")
# Replace 'sample_id' with the actual name of the sample column in your methylation data if different
clin_df = pd.read_csv('../data/processed/clinical_hnsc_all.csv')

Loading datasets...


In [3]:
clin_df

,sample,tissue_type.samples,alcohol_history.exposures,immune_subtypes
0,TCGA-BB-4227-01A,Tumor,Yes,Wound Healing (Immune C1)
1,TCGA-CR-6478-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2)
2,TCGA-BB-4228-01A,Tumor,Yes,Wound Healing (Immune C1)
3,TCGA-BA-4075-01A,Tumor,Yes,NaN
4,TCGA-CX-7082-01A,Tumor,Yes,NaN
...,...,...,...,...
575,TCGA-CV-7242-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2)
576,TCGA-TN-A7HL-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2)
577,TCGA-CV-A6JZ-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2)
578,TCGA-CV-5444-11A,Normal,Yes,NaN


In [4]:
meth_df = pd.read_csv('../data/processed/methylation_hnsc_all.csv', index_col=0)
meth_df

,cg00000029,cg00000108,cg00000109,cg00000165,cg00000236,cg00000292,cg00000321,cg00000363,cg00000622,cg00000658,...,ctl_70664314,ctl_70700334,ctl_71670310,ctl_71678368,ctl_71718498,ctl_72748406,ctl_73635489,ctl_73784382,ctl_73794434,ctl_74666473
TCGA-F7-A61V-01A,0.169494,0.969504,0.901731,0.575925,0.884583,0.443790,0.527111,0.201535,0.016330,0.899096,...,0.024935,0.074955,0.934347,0.074081,0.109095,0.123795,0.055536,0.893430,0.924984,0.958676
TCGA-CV-7407-01A,0.271910,0.968134,0.955765,0.643235,0.934308,0.772394,0.736274,0.313491,0.014606,0.832098,...,0.097556,0.054384,0.965121,0.051201,0.218709,0.076075,0.070833,0.929056,0.972468,0.952202
TCGA-UF-A7JO-01A,0.417004,0.955937,0.921865,0.488987,0.906538,0.757155,0.564353,0.325092,0.015789,0.843975,...,0.028748,0.091012,0.960164,0.055483,0.166570,0.120988,0.063425,0.931578,0.951702,0.964257
TCGA-CV-6948-01A,0.128467,0.968904,0.910797,0.860205,0.921293,0.547383,0.674936,0.230438,0.012691,0.889466,...,0.039361,0.055663,0.960194,0.065371,0.177939,0.097036,0.088908,0.936490,0.955121,0.957882
TCGA-BB-7864-01A,0.171842,0.958299,0.945290,0.723734,0.904258,0.461953,0.285278,0.221449,0.013150,0.889051,...,0.023655,0.063635,0.968602,0.053799,0.178144,0.092968,0.052888,0.955130,0.970614,0.968689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-CV-7432-01A,0.485641,0.954409,0.937185,0.734791,0.917020,0.736587,0.466652,0.435981,0.015868,0.806900,...,0.129434,0.072841,0.966988,0.078038,0.219387,0.115900,0.061564,0.921702,0.956045,0.957312
TCGA-CR-6471-01A,0.393504,0.957079,0.929260,0.705929,0.926273,0.733265,0.559355,0.275917,0.017546,0.878021,...,0.070154,0.101726,0.949204,0.053345,0.188774,0.124597,0.066099,0.938521,0.953174,0.978739
TCGA-MT-A67A-01A,0.606525,0.958249,0.923488,0.499184,0.919832,0.665014,0.574861,0.346464,0.012115,0.844069,...,0.018265,0.086160,0.967759,0.055477,0.086231,0.102222,0.062569,0.944587,0.963469,0.959490
TCGA-H7-7774-01A,0.561132,0.973065,0.924097,0.692248,0.930900,0.749997,0.654586,0.266455,0.015712,0.873654,...,0.080932,0.058969,0.967671,0.056863,0.184611,0.072771,0.062440,0.944116,0.960702,0.952825


In [5]:
# Extract sample IDs from the index
meth_samples = meth_df.index.values

# Extract the raw methylation features (all columns)
meth_features = meth_df.values

# (Optional but highly recommended) Scale the data before UMAP and Clustering
print("Scaling data...")
scaler = StandardScaler()
meth_features_scaled = scaler.fit_transform(meth_features)

# 2. Dimensionality Reduction (UMAP from 406601 to 2)
print("Running UMAP (n=2)...")
# Note: Using the scaled features here, but you can change to meth_features if you prefer raw
reducer_1 = umap.UMAP(n_components=2, random_state=42)
meth_umap_2d = reducer_1.fit_transform(meth_features_scaled)

print("Running UMAP (n=3)...")
# Note: Using the scaled features here, but you can change to meth_features if you prefer raw
reducer_2 = umap.UMAP(n_components=3, random_state=42)
meth_umap_3d = reducer_2.fit_transform(meth_features_scaled)

# 3. Clustering 

N_CLUSTERS_2 = 2

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP 2d data...")
kmeans_umap_2d = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_umap_2d_labels_k2 = kmeans_umap_2d.fit_predict(meth_umap_2d)

# Feature 2: K-means (K=6) on Raw data
print("Running K-Means on Raw data (This may take a while)...")
kmeans_raw = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_raw_labels_k2 = kmeans_raw.fit_predict(meth_features_scaled)

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP 3d data...")
kmeans_umap_3d = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_umap_3d_labels_k2 = kmeans_umap_3d.fit_predict(meth_umap_3d)

# Feature 5: Hierarchical (Agglomerative) on UMAP data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on UMAP 2d data...")
# 'ward' linkage minimizes the variance of the clusters being merged
hierarchical_umap_2d = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_umap_2d_labels_k2 = hierarchical_umap_2d.fit_predict(meth_umap_2d)

# Feature 6: Hierarchical (Agglomerative) on UMAP 3d data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on UMAP 3d data...")
hierarchical_umap_3d = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_umap_3d_labels_k2 = hierarchical_umap_3d.fit_predict(meth_umap_3d)

# Feature 7: Hierarchical (Agglomerative) on Raw data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on Raw data...")
hierarchical_raw = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_raw_labels_k2 = hierarchical_raw.fit_predict(meth_features_scaled)

N_CLUSTERS_6 = 6

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP 2d data...")
kmeans_umap_2d = KMeans(n_clusters=N_CLUSTERS_6, random_state=42)
kmeans_umap_2d_labels_k6 = kmeans_umap_2d.fit_predict(meth_umap_2d)

# Feature 2: K-means (K=6) on Raw data
print("Running K-Means on Raw data (This may take a while)...")
kmeans_raw = KMeans(n_clusters=N_CLUSTERS_6, random_state=42)
kmeans_raw_labels_k6 = kmeans_raw.fit_predict(meth_features_scaled)

# Feature 1: K-means (K=6) on UMAP data
print("Running K-Means on UMAP 3d data...")
kmeans_umap_3d = KMeans(n_clusters=N_CLUSTERS_6, random_state=42)
kmeans_umap_3d_labels_k6 = kmeans_umap_3d.fit_predict(meth_umap_3d)

# Feature 5: Hierarchical (Agglomerative) on UMAP data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_6}) on UMAP 2d data...")
# 'ward' linkage minimizes the variance of the clusters being merged
hierarchical_umap_2d = AgglomerativeClustering(n_clusters=N_CLUSTERS_6, linkage='ward')
hierarchical_umap_2d_labels_k6 = hierarchical_umap_2d.fit_predict(meth_umap_2d)

# Feature 6: Hierarchical (Agglomerative) on UMAP 3d data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_6}) on UMAP 3d data...")
hierarchical_umap_3d = AgglomerativeClustering(n_clusters=N_CLUSTERS_6, linkage='ward')
hierarchical_umap_3d_labels_k6 = hierarchical_umap_3d.fit_predict(meth_umap_3d)

# Feature 7: Hierarchical (Agglomerative) on Raw data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_6}) on Raw data...")
hierarchical_raw = AgglomerativeClustering(n_clusters=N_CLUSTERS_6, linkage='ward')
hierarchical_raw_labels_k6 = hierarchical_raw.fit_predict(meth_features_scaled)

Scaling data...
Running UMAP (n=2)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running UMAP (n=3)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running K-Means on UMAP 2d data...
Running K-Means on Raw data (This may take a while)...


...

/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Running K-Means on UMAP 2d data...
Running K-Means on Raw data (This may take a while)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Running K-Means on UMAP 3d data...
Running Hierarchical Clustering (K=6) on UMAP 2d data...
Running Hierarchical Clustering (K=6) on UMAP 3d data...
Running Hierarchical Clustering (K=6) on Raw data...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [6]:
# # Feature 3: DBSCAN on UMAP data
# print("Running DBSCAN on UMAP data...")
# # Note: You will need to tune eps and min_samples to find your desired 6 clusters
# dbscan_umap = DBSCAN(eps=0.5, min_samples=5) 
# dbscan_umap_labels = dbscan_umap.fit_predict(meth_umap)

# # Feature 4: DBSCAN on Raw data
# print("Running DBSCAN on Raw data (This will require high memory)...")
# # Note: Distances in high dimensions are very large; eps usually needs to be significantly higher here.
# dbscan_raw = DBSCAN(eps=5.0, min_samples=5) 
# dbscan_raw_labels = dbscan_raw.fit_predict(meth_features_scaled)


In [7]:
 #4. Compile the new features into a DataFrame
print("Compiling results...")
cluster_results = pd.DataFrame({
    'sample': meth_samples,
    'kmeans_umap_2_k2': kmeans_umap_2d_labels_k2,
    'kmeans_umap_3_k2': kmeans_umap_3d_labels_k2,
    'kmeans_raw_k2': kmeans_raw_labels_k2,
    'hierarchical_umap_2_k2': hierarchical_umap_2d_labels_k2,
    'hierarchical_umap_3_k2': hierarchical_umap_3d_labels_k2,
    'hierarchical_raw_k2': hierarchical_raw_labels_k2,
    'kmeans_umap_2_k6': kmeans_umap_2d_labels_k6,
    'kmeans_umap_3_k6': kmeans_umap_3d_labels_k6,
    'kmeans_raw_k6': kmeans_raw_labels_k6,
    'hierarchical_umap_2_k6': hierarchical_umap_2d_labels_k6,
    'hierarchical_umap_3_k6': hierarchical_umap_3d_labels_k6,
    'hierarchical_raw_k6': hierarchical_raw_labels_k6
})

# 5. Merge with the clinical dataset
print("Merging with clinical data...")
# Merging on 'sample' column. Using an inner or left join to ensure alignment
final_df = pd.merge(clin_df, cluster_results, on='sample', how='left')

# 6. Save the new dataset
output_file = '../data/processed/clinical_hnsc_all_clustered.csv'

Compiling results...
Merging with clinical data...


In [8]:
final_df

,sample,tissue_type.samples,alcohol_history.exposures,immune_subtypes,kmeans_umap_2_k2,kmeans_umap_3_k2,kmeans_raw_k2,hierarchical_umap_2_k2,hierarchical_umap_3_k2,hierarchical_raw_k2,kmeans_umap_2_k6,kmeans_umap_3_k6,kmeans_raw_k6,hierarchical_umap_2_k6,hierarchical_umap_3_k6,hierarchical_raw_k6
0,TCGA-BB-4227-01A,Tumor,Yes,Wound Healing (Immune C1),0,1,0,0,1,0,4,4,1,4,1,0
1,TCGA-CR-6478-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2),0,1,1,0,1,0,2,1,3,1,0,1
2,TCGA-BB-4228-01A,Tumor,Yes,Wound Healing (Immune C1),0,1,1,0,1,0,2,1,2,1,2,1
3,TCGA-BA-4075-01A,Tumor,Yes,NaN,0,1,1,0,1,0,2,4,3,1,0,1
4,TCGA-CX-7082-01A,Tumor,Yes,NaN,0,1,0,0,1,0,5,4,1,4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,TCGA-CV-7242-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2),0,0,0,0,0,1,0,0,0,5,5,2
576,TCGA-TN-A7HL-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2),0,1,1,0,1,0,2,1,4,1,2,4
577,TCGA-CV-A6JZ-01A,Tumor,Yes,IFN-gamma Dominant (Immune C2),1,1,1,0,1,0,1,3,3,2,0,4
578,TCGA-CV-5444-11A,Normal,Yes,NaN,1,1,1,1,0,0,3,2,5,3,3,5


In [9]:
final_df.to_csv(output_file, index=False)
print(f"Process complete! Saved new dataset to {output_file}")

Process complete! Saved new dataset to ../data/processed/clinical_hnsc_all_clustered.csv


In [10]:
# import pandas as pd
# from sklearn.metrics import adjusted_rand_score

# # 1. Load the dataset (if you are running this in a new script)
# # final_df = pd.read_csv('clinical_hnsc_all_sb_clustered.csv')

# # 2. Drop rows where the ground truth is missing
# # ARI cannot evaluate samples that do not have a known immune subtype
# #valid_df = final_df.dropna(subset=['immune_subtypes']).copy()
# #valid_df = final_df.dropna(subset=['Subtype_mRNA']).copy()
# valid_df = final_df.dropna(subset=['tissue_type.samples']).copy()

# # Extract the true labels
# #true_labels = valid_df['immune_subtypes']
# #true_labels = valid_df['Subtype_mRNA']
# true_labels = valid_df['tissue_type.samples']

# # 3. Define the clustering columns to evaluate
# cluster_columns = [
#     'kmeans_umap_50_k6',
#     'kmeans_raw_k6',
#     'dbscan_umap_50',
#     'dbscan_raw'
# ]

# # 4. Compute and print the ARI for each feature
# print("Adjusted Rand Index (ARI) Scores:")
# print("-" * 40)

# for col in cluster_columns:
#     pred_labels = valid_df[col]
#     ari_score = adjusted_rand_score(true_labels, pred_labels)
#     print(f"{col:<20} : {ari_score:.4f}")

In [12]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# 1. Load the dataset (if you are running this in a new script)
# final_df = pd.read_csv('clinical_hnsc_all_sb_clustered.csv')

# 2. Define columns and drop ALL rows containing NaNs in these specific columns
cluster_columns = [
    'kmeans_umap_2_k6',
    'kmeans_umap_3_k6',
    'kmeans_raw_k6',
    'hierarchical_umap_2_k6',
    'hierarchical_umap_3_k6',
    'hierarchical_raw_k6'
]
target_column = 'immune_subtypes'  # Change this to the actual name of your ground truth column

columns_to_check = [target_column] + cluster_columns
valid_df = final_df.dropna(subset=columns_to_check).copy()

# Extract the true labels
true_labels = valid_df[target_column]

# 3. Compute and print the ARI for each feature
print(f"Adjusted Rand Index (ARI) Scores (Evaluated on {len(valid_df)} samples):")
print("-" * 55)

for col in cluster_columns:
    pred_labels = valid_df[col]
    ari_score = adjusted_rand_score(true_labels, pred_labels)
    print(f"{col:<25} : {ari_score:.4f}")

Adjusted Rand Index (ARI) Scores (Evaluated on 514 samples):
-------------------------------------------------------
kmeans_umap_2_k6          : 0.0491
kmeans_umap_3_k6          : 0.0234
kmeans_raw_k6             : 0.0377
hierarchical_umap_2_k6    : 0.0023
hierarchical_umap_3_k6    : 0.0289
hierarchical_raw_k6       : 0.0331


In [13]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# 1. Load the dataset (if you are running this in a new script)
# final_df = pd.read_csv('clinical_hnsc_all_sb_clustered.csv')

# 2. Define columns and drop ALL rows containing NaNs in these specific columns
cluster_columns = [
    'kmeans_umap_2_k2',
    'kmeans_umap_3_k2',
    'kmeans_raw_k2',
    'hierarchical_umap_2_k2',
    'hierarchical_umap_3_k2',
    'hierarchical_raw_k2'
]
target_column = 'tissue_type.samples'  # Change this to the actual name of your ground truth column

columns_to_check = [target_column] + cluster_columns
valid_df = final_df.dropna(subset=columns_to_check).copy()

# Extract the true labels
true_labels = valid_df[target_column]

# 3. Compute and print the ARI for each feature
print(f"Adjusted Rand Index (ARI) Scores (Evaluated on {len(valid_df)} samples):")
print("-" * 55)

for col in cluster_columns:
    pred_labels = valid_df[col]
    ari_score = adjusted_rand_score(true_labels, pred_labels)
    print(f"{col:<25} : {ari_score:.4f}")

Adjusted Rand Index (ARI) Scores (Evaluated on 580 samples):
-------------------------------------------------------
kmeans_umap_2_k2          : 0.0813
kmeans_umap_3_k2          : -0.0812
kmeans_raw_k2             : -0.0783
hierarchical_umap_2_k2    : 0.9351
hierarchical_umap_3_k2    : 0.4468
hierarchical_raw_k2       : -0.0835


In [14]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# 1. Load the dataset (if you are running this in a new script)
# final_df = pd.read_csv('clinical_hnsc_all_sb_clustered.csv')

# 2. Define columns and drop ALL rows containing NaNs in these specific columns
cluster_columns = [
    'kmeans_umap_2_k2',
    'kmeans_umap_3_k2',
    'kmeans_raw_k2',
    'hierarchical_umap_2_k2',
    'hierarchical_umap_3_k2',
    'hierarchical_raw_k2'
]
target_column = 'alcohol_history.exposures'  # Change this to the actual name of your ground truth column

columns_to_check = [target_column] + cluster_columns
valid_df = final_df.dropna(subset=columns_to_check).copy()

# Extract the true labels
true_labels = valid_df[target_column]

# 3. Compute and print the ARI for each feature
print(f"Adjusted Rand Index (ARI) Scores (Evaluated on {len(valid_df)} samples):")
print("-" * 55)

for col in cluster_columns:
    pred_labels = valid_df[col]
    ari_score = adjusted_rand_score(true_labels, pred_labels)
    print(f"{col:<25} : {ari_score:.4f}")

Adjusted Rand Index (ARI) Scores (Evaluated on 580 samples):
-------------------------------------------------------
kmeans_umap_2_k2          : 0.0002
kmeans_umap_3_k2          : -0.0040
kmeans_raw_k2             : 0.0077
hierarchical_umap_2_k2    : -0.0140
hierarchical_umap_3_k2    : -0.0126
hierarchical_raw_k2       : 0.0056


In [15]:
# for col in valid_df.columns:
#     print(f"--- {col} ---")
#     print(valid_df[col].value_counts())
#     print("\n")